# CPSC 483 – Final Evaluation: Models, Metrics, and Figures

This notebook trains every model with the same procedure, computes all metrics, and saves the tables and figures used in the paper.

| Task | Models | Targets | Baselines |
|---|---|---|---|
| A (daily classification) | Logistic Regression, Random Forest | `hot_day`, `hot_night` | Always "no", Climatology |
| B (yearly regression) | Linear Regression, Random Forest | `hot_day_count`, `hot_night_count` | Station climatology |

**Procedure**
1. Train on 1950–2005.
2. Task A only: choose the decision threshold that maximizes F1 on the validation period (2006–2015).
3. Retrain on 1950–2015.
4. Evaluate once on the test period (2016–2025).

We also retrain every model **without the `year` feature** to see how a time trend affects each model.

Outputs are saved to `results/` (CSV and LaTeX tables), `figures/` (PDF and PNG), and `models/` (final models as `.sav`).

# 0. Setup

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, precision_recall_curve,
                             r2_score, mean_absolute_error)

SEED = 42
np.random.seed(SEED)

DATA_FILE = "Final_SOCal_dataset.csv"
TRAIN_END = 2005      # train: 1950–2005
VAL_END = 2015        # validation: 2006–2015, test: 2016–2025

FIG_DIR = Path("figures")
RES_DIR = Path("results")
MODEL_DIR = Path("models")
for d in (FIG_DIR, RES_DIR, MODEL_DIR):
    d.mkdir(exist_ok=True)

plt.rc("font", size=8)
plt.rc("axes", titlesize=9, labelsize=8)
plt.rc("legend", fontsize=7)
plt.rc("xtick", labelsize=7)
plt.rc("ytick", labelsize=7)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

# 1. Load data

In [ ]:
data = pd.read_csv(DATA_FILE)

STATION_ORDER = ["LAX", "LA_Downtown", "Santa_Ana", "Riverside", "Palm_Springs"]
STATION_LABEL = {"LAX": "LAX", "LA_Downtown": "LA Downtown", "Santa_Ana": "Santa Ana",
                 "Riverside": "Riverside", "Palm_Springs": "Palm Springs"}

A_FEATURES = ["tmin_lag1", "dtr_lag1", "dayofyear", "days_since_rain",
              "prcp_lag1", "prcp_sum30", "oni_lag", "station_code", "year"]
A_TARGETS = ["hot_day", "hot_night"]

B_FEATURES = ["prev_hot_nights", "prev_mean_tmin", "avg5_hot_days",
              "prev_max_spell_days", "station_code", "year"]
B_TARGETS = ["hot_day_count", "hot_night_count"]

FEATURE_LABEL = {
    "tmin_lag1": "Previous-day low", "dtr_lag1": "Previous-day range",
    "dayofyear": "Day of year", "days_since_rain": "Days since rain",
    "prcp_lag1": "Previous-day rain", "prcp_sum30": "30-day rain",
    "oni_lag": "ONI", "station_code": "Station", "year": "Year",
}
TARGET_LABEL = {"hot_day": "Extreme heat days", "hot_night": "Extreme heat nights",
                "hot_day_count": "Extreme heat days", "hot_night_count": "Extreme heat nights"}

# Task B: one row per station-year (the Task B columns repeat on every day of a year)
yearly = data.drop_duplicates(subset=["station", "year"]).copy()
print("Daily rows:", len(data), "| Station-years:", len(yearly))

# 2. Split by year

In [ ]:
def split(df):
    train = df[df["year"] <= TRAIN_END]
    val = df[(df["year"] > TRAIN_END) & (df["year"] <= VAL_END)]
    full_train = df[df["year"] <= VAL_END]          # train + validation, for the final model
    test = df[df["year"] > VAL_END]
    return train, val, full_train, test

# 3. Task A – Daily classification

**Models**
- **Logistic Regression**: numerical features are scaled and the station is one-hot encoded.
- **Random Forest**: 100 trees, features used as they are.
- Both use balanced class weights, because extreme heat is rare.

**Baselines**
- **Always "no"**: always predicts "no extreme heat".
- **Climatology**: the historical frequency of extreme heat for the same station and day of year, averaged over a ±7-day window so that each value uses several hundred past days.

In [ ]:
A_NUMERIC = [f for f in A_FEATURES if f != "station_code"]

def make_logistic():
    prep = make_column_transformer(
        (StandardScaler(), A_NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), ["station_code"]),
    )
    return make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced"))

def make_rf_classifier():
    return RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                  n_jobs=-1, random_state=SEED)

class Climatology:
    """Historical frequency of extreme heat per station and day of year (±window days)."""
    def __init__(self, window=7):
        self.window = window

    def fit(self, X, y):
        df = pd.DataFrame({"station_code": X["station_code"].values,
                           "dayofyear": X["dayofyear"].values, "y": np.asarray(y)})
        grid = (df.groupby(["station_code", "dayofyear"])["y"].agg(["sum", "count"])
                  .unstack("station_code").reindex(range(1, 367)).fillna(0))
        k = 2 * self.window + 1
        hits = grid["sum"].rolling(k, center=True, min_periods=1).sum()
        days = grid["count"].rolling(k, center=True, min_periods=1).sum()
        self.table_ = (hits / days.replace(0, np.nan)).stack().rename("p")
        self.station_rate_ = df.groupby("station_code")["y"].mean()
        return self

    def predict_proba(self, X):
        key = pd.MultiIndex.from_arrays([X["dayofyear"].values, X["station_code"].values])
        p = self.table_.reindex(key).to_numpy()
        fallback = X["station_code"].map(self.station_rate_).to_numpy()
        p = np.where(np.isnan(p), fallback, p)
        return np.column_stack([1 - p, p])

A_MODELS = {
    "Climatology": lambda: Climatology(window=7),
    "Logistic Regression": make_logistic,
    "Random Forest": make_rf_classifier,
}

In [ ]:
def best_threshold(y_true, proba):
    """Threshold that maximizes F1."""
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]

def classification_row(task, target, name, y_true, y_pred, proba, threshold):
    return {
        "Target": TARGET_LABEL[target], "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "PR-AUC": average_precision_score(y_true, proba),
    }

def make_logistic_for(features):
    numeric = [f for f in features if f != "station_code"]
    prep = make_column_transformer(
        (StandardScaler(), numeric),
        (OneHotEncoder(handle_unknown="ignore"), ["station_code"]),
    )
    return make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced"))

def run_task_a(features, include_baselines=True):
    # Train every Task A model with the same procedure.
    models = {"Climatology": lambda: Climatology(window=7),
              "Logistic Regression": lambda: make_logistic_for(features),
              "Random Forest": make_rf_classifier}
    if not include_baselines:
        models.pop("Climatology")
    rows, probas, finals = [], {}, {}
    for target in A_TARGETS:
        df = data.dropna(subset=features + [target])
        train, val, full_train, test = split(df)
        y_test = test[target].astype(int)
        if include_baselines:
            print(f"{target}: train {len(train)}, validation {len(val)}, test {len(test)}, "
                  f"test positive rate {y_test.mean():.3f}")
            zeros = np.zeros(len(test))
            rows.append(classification_row("A", target, 'Always "no"', y_test, zeros, zeros, np.nan))
        for name, make in models.items():
            # 1) fit on 1950–2005, choose threshold on 2006–2015
            model = make().fit(train[features], train[target].astype(int))
            threshold = best_threshold(val[target].astype(int),
                                       model.predict_proba(val[features])[:, 1])
            # 2) refit on 1950–2015, evaluate on 2016–2025
            model = make().fit(full_train[features], full_train[target].astype(int))
            proba = model.predict_proba(test[features])[:, 1]
            pred = (proba >= threshold).astype(int)
            rows.append(classification_row("A", target, name, y_test, pred, proba, threshold))
            probas[(target, name)] = (y_test.to_numpy(), proba)
            finals[(target, name)] = (model, threshold)
    return pd.DataFrame(rows), probas, finals

task_a_results, a_test_proba, a_final_models = run_task_a(A_FEATURES)
task_a_results.round(3)

# 4. Task B – Yearly regression

**Models**
- **Linear Regression**: the station is one-hot encoded.
- **Random Forest**: 100 trees.

**Baseline**
- **Station climatology**: each station's average count over 1950–2015.

Task B has no threshold or setting to choose, so the models are trained directly on 1950–2015.

In [ ]:
B_NUMERIC = [f for f in B_FEATURES if f != "station_code"]

def make_linear():
    prep = make_column_transformer(
        ("passthrough", B_NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), ["station_code"]),
    )
    return make_pipeline(prep, LinearRegression())

def make_rf_regressor():
    return RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)

B_MODELS = {"Linear Regression": make_linear, "Random Forest": make_rf_regressor}

def regression_row(target, name, y_true, y_pred):
    return {
        "Target": TARGET_LABEL[target], "Model": name,
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "Mean bias": np.mean(y_pred - y_true),
    }

def make_linear_for(features):
    numeric = [f for f in features if f != "station_code"]
    prep = make_column_transformer(
        ("passthrough", numeric),
        (OneHotEncoder(handle_unknown="ignore"), ["station_code"]),
    )
    return make_pipeline(prep, LinearRegression())

def run_task_b(features, include_baselines=True):
    # Train every Task B model on 1950–2015 and evaluate on 2016–2025.
    models = {"Linear Regression": lambda: make_linear_for(features),
              "Random Forest": make_rf_regressor}
    rows, test_preds, finals = [], {}, {}
    for target in B_TARGETS:
        df = yearly.dropna(subset=features + [target])
        _, _, full_train, test = split(df)
        y_test = test[target].to_numpy()
        preds = test[["name", "year", target]].copy()
        if include_baselines:
            print(f"{target}: train {len(full_train)}, test {len(test)}")
            station_mean = full_train.groupby("station_code")[target].mean()
            clim = test["station_code"].map(station_mean).to_numpy()
            rows.append(regression_row(target, "Station climatology", y_test, clim))
            preds["Station climatology"] = clim
        for name, make in models.items():
            model = make().fit(full_train[features], full_train[target])
            p = model.predict(test[features])
            rows.append(regression_row(target, name, y_test, p))
            preds[name] = p
            finals[(target, name)] = model
        test_preds[target] = preds
    return pd.DataFrame(rows), test_preds, finals

task_b_results, b_test_pred, b_final_models = run_task_b(B_FEATURES)
task_b_results.round(3)

# 5. Additional analysis – models without `year`

Every model is retrained with the same procedure but without the `year` feature.
Tree models cannot extend a trend beyond the training years, so `year` can affect them differently from the linear models.

In [ ]:
A_NO_YEAR = [f for f in A_FEATURES if f != "year"]
B_NO_YEAR = [f for f in B_FEATURES if f != "year"]

a_noyear, _, _ = run_task_a(A_NO_YEAR, include_baselines=False)
b_noyear, _, _ = run_task_b(B_NO_YEAR, include_baselines=False)

a_cmp = (task_a_results.merge(a_noyear, on=["Target", "Model"], suffixes=("", " (no year)"))
         [["Target", "Model", "F1", "F1 (no year)", "PR-AUC", "PR-AUC (no year)"]])
b_cmp = (task_b_results.merge(b_noyear, on=["Target", "Model"], suffixes=("", " (no year)"))
         [["Target", "Model", "R2", "R2 (no year)", "Mean bias", "Mean bias (no year)"]])
display(a_cmp.round(3))
display(b_cmp.round(3))

# 6. Save the final models

The final models (trained on 1950–2015) are saved with `pickle`, then loaded back and scored on the test period.
For Task A, the decision threshold chosen on the validation period is saved in `results/taskA_thresholds.csv`.
`score()` gives accuracy for the classifiers (at the default 0.5 threshold) and R² for the regressors.

In [ ]:
FILE_NAME = {"Logistic Regression": "logistic", "Random Forest": "rf", "Linear Regression": "linear"}
saved = []
threshold_rows = []

for target in A_TARGETS:
    df = data.dropna(subset=A_FEATURES + [target])
    test = split(df)[3]
    for name in ["Logistic Regression", "Random Forest"]:
        model, threshold = a_final_models[(target, name)]
        path = MODEL_DIR / f"taskA_{FILE_NAME[name]}_{target}.sav"
        pickle.dump(model, open(path, "wb"))
        saved.append((path, test[A_FEATURES], test[target].astype(int)))
        threshold_rows.append({"Target": target, "Model": name, "Threshold": threshold})

for target in B_TARGETS:
    df = yearly.dropna(subset=B_FEATURES + [target])
    test = split(df)[3]
    for name in ["Linear Regression", "Random Forest"]:
        path = MODEL_DIR / f"taskB_{FILE_NAME[name]}_{target}.sav"
        pickle.dump(b_final_models[(target, name)], open(path, "wb"))
        saved.append((path, test[B_FEATURES], test[target]))

pd.DataFrame(threshold_rows).to_csv(RES_DIR / "taskA_thresholds.csv", index=False)

for path, X_test, y_test in saved:
    loaded_model = pickle.load(open(path, "rb"))
    print(f"{path.name:40s} {loaded_model.score(X_test, y_test):.3f}")

# 7. Figures

## Figure 1 – Yearly extreme heat days and nights

5-year centered moving average per station. The shaded area is the test period.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4), sharey=True)
for ax, target in zip(axes, B_TARGETS):
    for st in STATION_ORDER:
        s = (yearly[yearly["name"] == st].set_index("year")[target]
             .reindex(range(1950, 2026)))
        ax.plot(s.index, s.rolling(5, center=True, min_periods=3).mean(),
                label=STATION_LABEL[st], linewidth=1.2)
    ax.axvspan(VAL_END + 0.5, 2025.5, color="grey", alpha=0.15, linewidth=0)
    ax.set_title(TARGET_LABEL[target])
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Count per May–Oct season")
axes[1].legend(loc="upper left", frameon=False)
save_fig("fig1_yearly_counts")

## Figure 2 – Precision–recall curves on the test period (Task A)

Numbers in the top-right corner are PR-AUC values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.6), sharey=True)
for ax, target in zip(axes, A_TARGETS):
    for name in A_MODELS:
        y_true, proba = a_test_proba[(target, name)]
        precision, recall, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        ax.plot(recall, precision, label=name, linewidth=1.2)
        ax.text(0.98, 0.98 - 0.08 * list(A_MODELS).index(name), f"{name}: {ap:.2f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=6.5,
                color=ax.lines[-1].get_color())
    ax.axhline(y_true.mean(), color="grey", linestyle="--", linewidth=0.8,
               label="Positive rate")
    ax.set_title(TARGET_LABEL[target])
    ax.set_xlabel("Recall")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Precision")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06))
save_fig("fig2_pr_curves")

## Figure 3 – Random Forest feature importance (Task A, final models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4), sharex=True)
for ax, target in zip(axes, A_TARGETS):
    model, _ = a_final_models[(target, "Random Forest")]
    imp = (pd.Series(model.feature_importances_, index=A_FEATURES)
           .rename(index=FEATURE_LABEL).sort_values())
    ax.barh(imp.index, imp.values)
    ax.set_title(TARGET_LABEL[target])
    ax.set_xlabel("Impurity-based importance")
    ax.grid(axis="x", alpha=0.3)
save_fig("fig3_feature_importance")

## Figure 4 – Observed vs. predicted extreme heat nights (Task B)

Observed counts from 1990 for context; predictions cover the test period (2016–2025).

In [ ]:
target = "hot_night_count"
preds = b_test_pred[target]
fig, axes = plt.subplots(1, 5, figsize=(7.2, 1.9), sharex=True)
for ax, st in zip(axes, STATION_ORDER):
    obs = yearly[(yearly["name"] == st) & (yearly["year"] >= 1990)].set_index("year")[target]
    obs = obs.reindex(range(1990, 2026))
    ax.plot(obs.index, obs.values, color="black", linewidth=1, label="Observed")
    p = preds[preds["name"] == st].sort_values("year")
    ax.plot(p["year"], p["Linear Regression"], marker="o", markersize=2.5,
            linewidth=1, label="Linear Regression")
    ax.plot(p["year"], p["Random Forest"], marker="s", markersize=2.5,
            linewidth=1, label="Random Forest")
    ax.axvline(VAL_END + 0.5, color="grey", linestyle="--", linewidth=0.8)
    ax.set_title(STATION_LABEL[st])
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Extreme heat nights")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, -0.12))
save_fig("fig4_taskB_observed_vs_predicted")

# 8. Export tables

In [ ]:
def to_latex_table(df, columns, caption, label, digits=3):
    """Simple IEEE-style LaTeX table (no extra packages needed)."""
    lines = [r"\begin{table}[htbp]", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\centering", r"\footnotesize",
             r"\begin{tabular}{|l|l|" + "c|" * (len(columns) - 2) + "}", r"\hline",
             " & ".join(rf"\textbf{{{c}}}" for c in columns) + r" \\", r"\hline"]
    for target, group in df.groupby("Target", sort=False):
        for i, (_, row) in enumerate(group.iterrows()):
            cells = [target if i == 0 else "", row["Model"]]
            for c in columns[2:]:
                v = row[c]
                cells.append("--" if pd.isna(v) else f"{v:.{digits}f}")
            lines.append(" & ".join(cells) + r" \\")
        lines.append(r"\hline")
    lines += [r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

a_cols = ["Target", "Model", "F1", "PR-AUC", "Precision", "Recall", "Accuracy"]
b_cols = ["Target", "Model", "R2", "MAE", "Mean bias"]

task_a_results.to_csv(RES_DIR / "taskA_results.csv", index=False)
task_b_results.to_csv(RES_DIR / "taskB_results.csv", index=False)

latex_a = to_latex_table(task_a_results, a_cols,
                         "Task A results on the test period (2016--2025)", "tab:taskA")
latex_b = to_latex_table(task_b_results, b_cols,
                         "Task B results on the test period (2016--2025)", "tab:taskB", digits=2)
latex_b = latex_b.replace(r"\textbf{R2}", r"\textbf{R$^2$}")

(RES_DIR / "taskA_table.tex").write_text(latex_a)
(RES_DIR / "taskB_table.tex").write_text(latex_b)
a_cmp.to_csv(RES_DIR / "taskA_no_year.csv", index=False)
b_cmp.to_csv(RES_DIR / "taskB_no_year.csv", index=False)
print(latex_a)
print()
print(latex_b)